# 🤗 HuggingFace in 5 Scenes

In [1]:
!pip install transformers datasets huggingface-hub torch sentencepiece sacremoses

## 1. HuggingFace Hub — The Amazon for AI Models

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
models = list(api.list_models(pipeline_tag="text-classification", limit=5, sort="downloads", direction=-1))

for m in models:
    print(m.id, "  |  downloads:", m.downloads)


## 2. Transformers Library — Tokenizer & Model

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
encoding  = tokenizer("HuggingFace is awesome!", return_tensors="pt")
tokens    = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])

for tok, tid in zip(tokens, encoding["input_ids"][0].tolist()):
    print(tok, "->", tid)


[CLS] -> 101
hugging -> 17662
##face -> 12172
is -> 2003
awesome -> 12476
! -> 999
[SEP] -> 102


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model     = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

inputs = tokenizer("HuggingFace is awesome!", return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits

probs = torch.softmax(logits, dim=-1)[0]
for idx, prob in enumerate(probs):
    print(model.config.id2label[idx], round(prob.item(), 4))


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2542.28it/s]


NEGATIVE 0.0001
POSITIVE 0.9999


## 3. Datasets Library

In [ ]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb", split="train")
print(ds)


In [ ]:
# Preview a few rows
for i in range(3):
    print("Label:", ds[i]["label"], " | Text:", ds[i]["text"][:80])


## 4. Pipelines — One-liner NLP

In [ ]:
from transformers import pipeline

pipe = pipeline("sentiment-analysis")
pipe(["I love this!", "This is terrible.", "It's okay."])


In [ ]:
pipe = pipeline("ner", aggregation_strategy="simple")
pipe("Sundar Pichai is the CEO of Google in Mountain View.")


In [ ]:
# `pipeline("translation_en_to_fr", ...)` was removed in transformers v5.
# Translation/seq2seq models like opus-mt are now loaded directly via Auto classes.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

translator_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
translator_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-fr")

inputs = translator_tokenizer("HuggingFace makes AI accessible to everyone.", return_tensors="pt")
generated = translator_model.generate(**inputs, max_new_tokens=40)
translator_tokenizer.batch_decode(generated, skip_special_tokens=True)


In [ ]:
# `pipeline("summarization", ...)` was removed in transformers v5.
# Summarization models like distilbart are now loaded directly via Auto classes.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

summarizer_tokenizer = AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-6-6")
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-6-6")

text = (
    "Artificial intelligence is intelligence demonstrated by machines. "
    "AI research focuses on intelligent agents that perceive their environment "
    "and take actions to maximize their goals. Modern AI powers search engines, "
    "recommendation systems, and self-driving cars."
)
inputs = summarizer_tokenizer(text, return_tensors="pt", truncation=True)
generated = summarizer_model.generate(**inputs, max_length=50, min_length=15)
summarizer_tokenizer.batch_decode(generated, skip_special_tokens=True)


## 5. Running Open-Source Models Locally

In [ ]:
# `pipeline("text2text-generation", ...)` was removed in transformers v5.
# flan-t5 is a seq2seq model, so load it directly via Auto classes instead.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

inputs = flan_tokenizer("Translate English to French: I love open source AI.", return_tensors="pt")
generated = flan_model.generate(**inputs, max_new_tokens=40)
flan_tokenizer.batch_decode(generated, skip_special_tokens=True)


In [ ]:
pipe = pipeline("text-generation", model="distilgpt2")
pipe("The future of artificial intelligence is", max_new_tokens=30)
